# 🧱 레고 카드메이커 - GPU 고화질 배경 제거

웹앱([레고 카드메이커](https://zed-leo-23.github.io/lego-card-maker/))의 배경 제거는 **어떤 기기에서도 동작하도록** 가벼운 모델을 브라우저 안에서 돌려요. 그래서 결과가 가끔 아쉬울 수 있어요.

이 노트북은 **Colab의 무료 GPU**로 훨씬 무거운(=훨씬 정교한) 배경 제거 모델을 돌려서, 더 깨끗한 누끼컷 PNG를 만들어줍니다. 만든 PNG를 다운로드해서 웹앱 1단계(사진 업로드)에 그대로 올리면, 이미 투명 배경이라 자동으로 2단계를 건너뛰고 바로 카드를 만들 수 있어요.

**사용 순서**
1. 메뉴 `런타임 → 런타임 유형 변경` 에서 **하드웨어 가속기를 GPU로 설정** (T4 등 아무거나 괜찮아요)
2. `런타임 → 모두 실행` 클릭
3. 마지막 셀에서 사진을 업로드하면 자동으로 처리 → 결과가 화면에 보이고 자동으로 다운로드됩니다
4. 다운로드된 PNG를 [웹앱](https://zed-leo-23.github.io/lego-card-maker/)에 업로드하세요

> Colab은 세션이 끊기거나(장시간 미사용 시) 무료 GPU 사용량에 제한이 있을 수 있어요. 이 노트북은 '가끔 고화질이 필요할 때 여는 도구'용이고, 평소 사용은 그냥 웹앱만으로 충분해요.

In [ ]:
#@title 1. 필요한 패키지 설치 (GPU 확인 포함)
import subprocess, sys

def run(cmd):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

try:
    import torch
    has_gpu = torch.cuda.is_available()
except Exception:
    has_gpu = False

if has_gpu:
    print('✅ GPU 감지됨 - onnxruntime-gpu를 설치합니다.')
    run('pip install -q "rembg[gpu]" onnxruntime-gpu')
else:
    print('⚠️ GPU가 감지되지 않았어요. 상단 메뉴에서 런타임 유형을 GPU로 바꾼 뒤 다시 실행해주세요.')
    print('   (CPU로도 동작은 하지만 느리고, 이 노트북을 쓰는 의미가 줄어들어요.)')
    run('pip install -q rembg onnxruntime')

from rembg import remove, new_session
# isnet-general-use: 웹앱이 쓰는 모델(isnet)의 더 무거운/정교한 버전 - 일반 사물에 품질이 좋아요.
session = new_session('isnet-general-use')
print('모델 준비 완료 ✅')

In [ ]:
#@title 2. 사진 업로드 → 배경 제거 → 미리보기 + 자동 다운로드
from google.colab import files
from PIL import Image
import io
import matplotlib.pyplot as plt

print('📷 레고 작품 사진을 선택하세요 (여러 장도 가능해요)')
uploaded = files.upload()

for filename, filedata in uploaded.items():
    input_img = Image.open(io.BytesIO(filedata)).convert('RGBA')

    result = remove(
        input_img,
        session=session,
        alpha_matting=True,             # 가장자리를 더 부드럽고 정교하게 다듬어요
        alpha_matting_foreground_threshold=240,
        alpha_matting_background_threshold=10,
        alpha_matting_erode_size=10,
    )

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(input_img)
    axes[0].set_title('원본')
    axes[0].axis('off')

    checker = Image.new('RGB', result.size, (235, 235, 235))
    checker.paste(result, mask=result.split()[3])
    axes[1].imshow(checker)
    axes[1].set_title('배경 제거 결과')
    axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    out_name = filename.rsplit('.', 1)[0] + '_cutout.png'
    result.save(out_name)
    print(f'💾 저장됨: {out_name} - 자동 다운로드를 시작합니다...')
    files.download(out_name)

print('\n✅ 완료! 다운로드된 PNG를 레고 카드메이커 웹앱(https://zed-leo-23.github.io/lego-card-maker/)에 업로드하세요.')